In [1]:
import numpy as np
from scipy import optimize
import scipy as sp

import matplotlib.pyplot as plt
import matplotlib.axes as axe
import pandas as pd
import datetime as dt
import gurobipy as gp
from gurobipy import GRB
import cvxpy as cp
import yaml

import random
from itertools import chain, combinations, product
import time

import glob
import os

plt.rcParams['text.usetex'] = True

# Functions

In [2]:
# def poly(alpha, x, p=4):
#     assert alpha.shape[0] == p+1,  "alpha should be a length-(p+1) vector."
#     assert np.all(alpha > 0.0), "All entries of alpha should be strictly positive."
#     assert x >= 0.0, "x should be non-negative."
    
#     return sum(alpha[k] * (x**k) for k in range(p))

# def poly_simplified(alpha, x, p=4):
#     assert alpha.shape[0] == p+1,  "alpha should be a length-(p+1) vector."
#     assert np.all(alpha > 0.0), "All entries of alpha should be strictly positive."
#     assert x >= 0.0, "x should be non-negative."
    
#     return alpha[0] + alpha[p-1] * (x**(p-1))

# def route_latency(alphas, xs, edges, p=4):
#     assert len(alphas.shape) == 2, "alphas should be a 2D array."
#     assert alphas.shape[0] == xs.shape[0], "The first dimension of alphas should equal the dimension of xs."
#     assert set(edges).issubset(range(alphas.shape[0])), "edges should be a subset of indices of edges"
    
#     return sum([poly(alphas[e, :], xs[e], p=4) for e in edges])


In [3]:
def reshape_vec(v):

    # print("length of v to reshape:", v.shape[0])

    len_v = v.shape[0]
    v_reshaped = v.reshape((len_v, 1))
    
    return v_reshaped


In [4]:
# DR: distributionally robust

def generate_R_from_edge_node_matrix(edge_node_matrix):
    assert len(edge_node_matrix.shape) == 2
    assert edge_node_matrix.shape[1] == 2
    
    num_edges = edge_node_matrix.shape[0]
    num_nodes = np.max(edge_node_matrix) + 1
    
    R = np.zeros((num_nodes - 1, num_edges))
    
    for node_index in range(R.shape[0]):
        for edge_index in range(R.shape[1]):
            if edge_node_matrix[edge_index][0] == node_index:
                R[node_index, edge_index] = 1
            elif edge_node_matrix[edge_index][1] == node_index:
                R[node_index, edge_index] = -1

    return R


def generate_Gamma(B, R):

    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    
    return inv_B - inv_B @ R.T @ R_inv_B_R_top @ R @ inv_B


def compute_eps_max(Gamma, B, R, alpha_mean_nominal, eta, delta):
    
    # print()
    # print("Gamma.shape", Gamma.shape)
    # print("alpha_mean_nominal.shape:", alpha_mean_nominal.shape)
    # print()
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec

    # Define and solve the CVXPY problem.
    eps = cp.Variable(1)
    tau = cp.Variable(num_edges)
    func = eps
    obj = cp.Maximize(func)
    constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
                 - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr += [tau >= 0.0]
    constr += [eps >= 0.0]
    prob = cp.Problem(obj, constr)
    prob.solve()

    # # Print result.
    # print()
    # print("eps_max:", eps.value[0])
    # print()

    return eps.value
    

def compute_opt_DR_toll_constrained(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec
    
    tau = cp.Variable(num_edges)
    func = eps * cp.norm(Gamma @ tau + offset_with_eta) \
            + cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
#     func = cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
    obj = cp.Minimize(func)
    constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
                 - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr += [tau >= 0.0]
    
    prob = cp.Problem(obj, constr)
    prob.solve()

    # # Print result.
    # print()
    # print("DR optimal toll:", tau.value)
    # print()
    # print("Worst-case DR latency:", prob.value)
    # print()
    
    return tau.value, prob.value

def compute_opt_DR_toll(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec
    
    tau = cp.Variable(num_edges)
    func = eps * cp.norm(Gamma @ tau + offset_with_eta) \
            + cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
#     func = cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
    obj = cp.Minimize(func)
    # constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
    #              - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr = []
    constr += [tau >= 0.0]
    
    prob = cp.Problem(obj, constr)
    prob.solve()

    # # Print result.
    # print()
    # print("DR optimal toll:", tau.value)
    # print()
    # print("Worst-case DR latency:", prob.value)
    # print()
    
    return tau.value, prob.value


def latency_eq_worst(B, R, alpha, tau, eta):
    num_nodes = R.shape[0] + 1
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    
    tau = reshape_vec(tau)
    alpha = reshape_vec(alpha)
    eta_vec = reshape_vec(eta_vec)
    
    BRR_matrix = np.block([[B, R.T], [R, np.zeros((num_nodes-1, num_nodes-1))]])

    return np.block([tau.T, eta_vec.T]) @ np.linalg.inv(BRR_matrix) @ np.block([[alpha + tau], [- eta_vec]])

    
def compute_worst_theta(B, R, tau, eta, alpha_mean_nominal, eps):

    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    
    BRR_matrix = np.block([[B, R.T], [R, np.zeros((num_nodes-1, num_nodes-1))]])
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))

    # print("tau.shape:", tau.shape)
    # print("eta_vec.shape", eta_vec.shape)
    
    tau_eta = np.append(tau, eta_vec)

    # print("np.block([np.identity(num_edges), np.zeros((num_nodes-1, num_edges))]).shape:", \
    #      np.block([np.identity(num_edges), np.zeros((num_nodes-1, num_edges))]).shape)
    # print("BRR_matrix.shape:", BRR_matrix.shape)
    # print("eta_vec.shape", eta_vec.shape)
    
    q_tau = np.block([np.identity(num_edges), np.zeros((num_edges, num_nodes-1))]) @ np.linalg.inv(BRR_matrix) @ tau_eta
    
    theta_worst = alpha_mean_nominal + eps * q_tau / np.linalg.norm(q_tau)
    
    return theta_worst

def sample_from_ball(center, radius, n_samples=1):
    center = np.asarray(center)
    d = center.shape[0]
    
    # Step 1: random directions
    x = np.random.normal(size=(n_samples, d))
    x /= np.linalg.norm(x, axis=1, keepdims=True)
    
    # Step 2: radii
    u = np.random.uniform(0, 1, size=(n_samples, 1))
    r = radius * (u ** (1.0 / d))
    
    # Step 3: combine
    samples = center + r * x
    
    return samples

def latency_eq_worst_under_distribution_shift(B, R, alpha_mean_nominal, tau, eta, eps):

    num_nodes = R.shape[0] + 1
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    
    tau = reshape_vec(tau)
    alpha_mean_nominal = reshape_vec(alpha_mean_nominal)
    eta_vec = reshape_vec(eta_vec)

    Gamma = generate_Gamma(B, R)
    B_inv = np.linalg.inv(B)
    R_B_inv_R_T = R @ B_inv @ R.T

    vec_temp = Gamma @ tau + B_inv @ R.T @ np.linalg.inv(R_B_inv_R_T) @ eta_vec

    return eps * np.linalg.norm(vec_temp) + tau.T @ Gamma @ tau + alpha_mean_nominal.T @ Gamma @ tau
    


In [5]:
# # a = np.array([1, 2, 3]).reshape((3, 1))
# # b = np.array([4, 5]).reshape((2, 1))

# a = np.array([1, 2, 3])
# b = np.array([4, 5])

# # np.block([[a], [b]])
# np.block([a, b])

# np.append([1, 2, 3], [2, 4])
# np.identity(3)
# np.block([np.identity(2), np.identity(2)])

# <font color='red'> To edit below: Transform Sioux Falls to Pigou

# Pigou Network

In [32]:
num_nodes = 1
num_edges = 2

In [33]:
beta = [1.5, 0.1]
B = np.diag(beta)
edge_node_matrix = np.array([[0, 1], [0, 1]])

alpha_mean_nominal = np.array([20.0, 30.0])

delta = 0.2
eta = 100.0

R = generate_R_from_edge_node_matrix(edge_node_matrix)
Gamma = generate_Gamma(B, R)
eps_max_original = compute_eps_max(Gamma, B, R, alpha_mean_nominal, eta, delta)[0]
eps_max_adjusted = max(min(eps_max_original, min(alpha_mean_nominal) - delta), 0.0)

print("eps_max_adjusted:", eps_max_adjusted)
print("eps_max_original:", eps_max_original)

# # Compute distributionally robust toll and corresponding worst-case latency (in theory):

# eps = 0.5 * eps_max_adjusted
# tau_opt, worst_case_ = compute_opt_DR_toll_constrained(Gamma, B, R, alpha_mean_nominal, eta, eps, delta)

# # Compute certainty equivalence toll and corresponding worst-case latency (in theory):



eps_max_adjusted: 19.8
eps_max_original: 39.8


## Main Code:

In [36]:
num_eps = 5
# eps_set = np.linspace(0.0, eps_max_adjusted, num = num_eps)
eps_set = np.linspace(0.0, eps_max_original, num = num_eps)
num_samples = 10000

worst_latency_array = np.zeros((num_eps, num_eps))
tau_array = np.zeros((num_eps, num_edges))
alpha_mean_array = np.zeros((num_eps, num_eps, num_edges))

# for eps_true, eps_hat in product(eps_set, repeat=2):
#     print(eps_true, eps_hat)

for i, eps_true in enumerate(eps_set):
    for j, eps_hat in enumerate(eps_set):
#         tau_eps_hat, _ = compute_opt_DR_toll_constrained(Gamma = Gamma,\
#                                              B = B,\
#                                              R = R,\
#                                              alpha_mean_nominal = alpha_mean_nominal,\
#                                              eta = eta,\
#                                              eps = eps_hat, \
#                                              delta = delta)

        tau_eps_hat, _ = compute_opt_DR_toll(Gamma = Gamma,\
                                     B = B,\
                                     R = R,\
                                     alpha_mean_nominal = alpha_mean_nominal,\
                                     eta = eta,\
                                     eps = eps_hat, \
                                     delta = delta)
        tau_array[j, :] = tau_eps_hat
        
        # print("tau_eps_hat:", tau_eps_hat)
        alpha_mean_eps = compute_worst_theta(B = B,\
                                             R = R,\
                                             tau = tau_eps_hat,\
                                             eta = eta,\
                                             alpha_mean_nominal = alpha_mean_nominal,\
                                             eps = eps_true)
        alpha_mean_array[i, j, :] = alpha_mean_eps
        alpha_vals = sample_from_ball(alpha_mean_eps, delta, n_samples = num_samples).T
        

        latency_array_temp = np.zeros(num_samples)
        for n in range(num_samples):
            latency_array_temp[n] = latency_eq_worst(B,\
                                                     R,\
                                                     alpha_vals[:, n],\
                                                     tau_eps_hat,\
                                                     eta)
        worst_latency_array[i, j] = np.mean(latency_array_temp)
        
        # worst_latency_array[i, j] = s
        # break


# def compute_opt_DR_toll(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
# def compute_worst_theta(B, R, tau, eta, alpha_mean_nominal, eps):
# def latency_eq_worst(B, R, alpha, tau, eta):


In [37]:
# print("alpha_vals.shape:", alpha_vals.shape)
worst_latency_array
# eps_set

array([[3859.41795957, 3870.66102079, 3900.84869179, 3945.00947428,
        3998.68389131],
       [4765.94602874, 4754.09380543, 4764.16206562, 4790.35207876,
        4828.7058812 ],
       [5672.49874956, 5637.51922921, 5627.32085023, 5635.8182704 ,
        5658.76409454],
       [6579.02235447, 6520.876509  , 6490.32014762, 6481.12278124,
        6488.77977447],
       [7485.44872845, 7404.25450971, 7353.35262943, 7326.57619417,
        7318.79348409]])

In [38]:
tau_array
# tau_array_gap_list = [np.mean(tau_array[eps_id_1, :] - tau_array[eps_id_2, :]) \
#                           for eps_id_1 in range(num_eps-1) for eps_id_2 in range(1, num_eps)]
# tau_array_gap_list

array([[ 5.00000344,  0.        ],
       [ 9.25478415, -0.        ],
       [13.15104351, -0.        ],
       [16.70230987, -0.        ],
       [19.93054681, -0.        ]])

In [39]:
for i, eps_true in enumerate(eps_set):
    for j, eps_hat in enumerate(eps_set):
        print(alpha_mean_array[i, j, :])

[20. 30.]
[20. 30.]
[20. 30.]
[20. 30.]
[20. 30.]
[21.02384681 39.89718332]
[21.34865715 39.85817549]
[21.65968183 39.81060427]
[21.95436556 39.75617524]
[22.23130133 39.69658674]
[22.04769362 49.79436664]
[22.69731429 49.71635097]
[23.31936366 49.62120855]
[23.90873111 49.51235048]
[24.46260265 49.39317348]
[23.07154043 59.69154997]
[24.04597144 59.57452646]
[24.97904549 59.43181282]
[25.86309667 59.26852571]
[26.69390398 59.08976022]
[24.09538724 69.58873329]
[25.39462858 69.43270194]
[26.63872732 69.2424171 ]
[27.81746222 69.02470095]
[28.9252053  68.78634696]


In [51]:
for i, eps_true in enumerate(eps_set):
    for j, eps_hat in enumerate(eps_set):
        print("eps_true, eps_hat:", eps_true, ",", eps_hat)
        
        tau_temp = tau_array[j, :]
        alpha_temp = alpha_mean_array[i, j, :]
        eta_vec = eta
        
        BR_block_inv = np.linalg.inv(np.block([[B, R.T], [R, 0]]))
        identity_zero_block = np.block([np.identity(2), np.zeros((2, 1))])
        vec_at_right = np.concatenate([-alpha_temp - tau_temp, [eta]])
        
        flow_Nash_expression = identity_zero_block @ BR_block_inv @ vec_at_right
        print(flow_Nash_expression)
        print()
        

eps_true, eps_hat: 0.0 , 0.0
[ 9.37499785 90.62500215]

eps_true, eps_hat: 0.0 , 9.95
[ 6.71575991 93.28424009]

eps_true, eps_hat: 0.0 , 19.9
[ 4.28059781 95.71940219]

eps_true, eps_hat: 0.0 , 29.849999999999998
[ 2.06105633 97.93894367]

eps_true, eps_hat: 0.0 , 39.8
[4.34082443e-02 9.99565918e+01]

eps_true, eps_hat: 9.95 , 0.0
[14.92083317 85.07916683]

eps_true, eps_hat: 9.95 , 9.95
[12.03420887 87.96579113]

eps_true, eps_hat: 9.95 , 19.9
[ 9.37492434 90.62507566]

eps_true, eps_hat: 9.95 , 29.849999999999998
[ 6.93718738 93.06281262]

eps_true, eps_hat: 9.95 , 39.8
[ 4.70921163 95.29078837]

eps_true, eps_hat: 19.9 , 0.0
[20.46666849 79.53333151]

eps_true, eps_hat: 19.9 , 9.95
[17.35265783 82.64734217]

eps_true, eps_hat: 19.9 , 19.9
[14.46925086 85.53074914]

eps_true, eps_hat: 19.9 , 29.849999999999998
[11.81331843 88.18668157]

eps_true, eps_hat: 19.9 , 39.8
[ 9.37501501 90.62498499]

eps_true, eps_hat: 29.849999999999998 , 0.0
[26.01250381 73.98749619]

eps_true, eps_hat: 

In [43]:
np.concatenate([[1, 2, 3], [4, 5]])

array([1, 2, 3, 4, 5])

In [23]:
worst_latency_array_from_observations = np.zeros((num_eps, num_eps))

for i, eps_true in enumerate(eps_set):
    for j, eps_hat in enumerate(eps_set):
        
        print("eps_true:", eps_true)
        print("eps_hat:", eps_hat)
        
        worst_latency_array_from_observations[i, j] = \
            latency_eq_worst_under_distribution_shift(B = B,\
                                                      R = R,\
                                                      alpha_mean_nominal = alpha_mean_nominal,\
                                                      tau = tau_array[j],\
                                                      eta = eta,
                                                      eps = eps_true)

# def latency_eq_worst_under_distribution_shift(B, R, alpha_mean_nominal, tau, eta, eps):

eps_true: 0.0
eps_hat: 0.0
eps_true: 0.0
eps_hat: 21.825
eps_true: 0.0
eps_hat: 43.65
eps_true: 0.0
eps_hat: 65.475
eps_true: 0.0
eps_hat: 87.3
eps_true: 21.825
eps_hat: 0.0
eps_true: 21.825
eps_hat: 21.825
eps_true: 21.825
eps_hat: 43.65
eps_true: 21.825
eps_hat: 65.475
eps_true: 21.825
eps_hat: 87.3
eps_true: 43.65
eps_hat: 0.0
eps_true: 43.65
eps_hat: 21.825
eps_true: 43.65
eps_hat: 43.65
eps_true: 43.65
eps_hat: 65.475
eps_true: 43.65
eps_hat: 87.3
eps_true: 65.475
eps_hat: 0.0
eps_true: 65.475
eps_hat: 21.825
eps_true: 65.475
eps_hat: 43.65
eps_true: 65.475
eps_hat: 65.475
eps_true: 65.475
eps_hat: 87.3
eps_true: 87.3
eps_hat: 0.0
eps_true: 87.3
eps_hat: 21.825
eps_true: 87.3
eps_hat: 43.65
eps_true: 87.3
eps_hat: 65.475
eps_true: 87.3
eps_hat: 87.3


In [22]:
worst_latency_array_from_observations

array([[   0.        ,    0.        ,    0.        ,    0.        ,
           0.        ],
       [1558.92857143, 1558.92857143, 1558.92857143, 1558.92857143,
        1558.92857143],
       [3117.85714286, 3117.85714286, 3117.85714286, 3117.85714286,
        3117.85714286],
       [4676.78571429, 4676.78571429, 4676.78571429, 4676.78571429,
        4676.78571429],
       [6235.71428571, 6235.71428571, 6235.71428571, 6235.71428571,
        6235.71428571]])

# <font color='red'> Red topic

# Scratch Work

In [ ]:
# Generate a random non-trivial linear program.
m = 15
n = 10
np.random.seed(1)
s0 = np.random.randn(m)
lamb0 = np.maximum(-s0, 0)
s0 = np.maximum(s0, 0)
x0 = np.random.randn(n)
A = np.random.randn(m, n)
b = A @ x0 + s0
c = -A.T @ lamb0

# Define and solve the CVXPY problem.
x = cp.Variable(n)
obj = cp.Minimize(c.T@x)
constr = [A @ x <= b]
prob = cp.Problem(obj, constr)
prob.solve()

# Print result.
print("\nThe optimal value is", prob.value)
print("A solution x is")
print(x.value)
print("A dual solution is")
print(prob.constraints[0].dual_value)

